# Layerwise — Depth 6

Launches the `layerwise` experiment for every seed-triple index in
`configs/layerwise_6layer.yaml` and summarises the completed runs. Results are written to
`results/layerwise/depth_6/seed_<N>/metrics.json`.


In [1]:
import os
import sys
from pathlib import Path

# Run from the project root regardless of the notebook's directory.
ROOT = Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")


'1'

## Run

Executes the runner once per seed-triple index, skipping
any seed whose `metrics.json` already exists.

In [2]:
import json
import subprocess
import time
from pathlib import Path

import yaml
from tqdm.notebook import tqdm

CONFIG = "configs/layerwise_6layer.yaml"
APPROACH = "layerwise"
DEVICE = "cpu"  # Use "cpu" or "auto" when needed.
results_dir = Path("results/layerwise/depth_6")

run_env = os.environ.copy()
run_env["TF_CPP_MIN_LOG_LEVEL"] = "2"
if DEVICE == "gpu":
    run_env["CUDA_VISIBLE_DEVICES"] = "0"
    run_env["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
elif DEVICE == "cpu":
    run_env["CUDA_VISIBLE_DEVICES"] = ""

config_data = yaml.safe_load(Path(CONFIG).read_text())
n_indices = int(config_data["seeds"]["seed_triples"])
total_updates = int(config_data["training"]["total_updates"])
n_layers = int(config_data["model"]["n_layers"])
per_stage_updates = total_updates // (n_layers + 1)
print(f"Layerwise depth 6 | device={DEVICE} | seeds={n_indices} | updates/seed={total_updates}")


def format_duration(seconds):
    seconds = max(0, int(seconds))
    return f"{seconds // 60}m {seconds % 60:02d}s"


def checkpoint_step(checkpoint_dir, metrics_path):
    """Infer progress from layerwise phase state and completed metrics."""
    if metrics_path.exists():
        return total_updates
    state_file = checkpoint_dir / "phase_state.json"
    if not state_file.exists():
        return 0
    state = json.loads(state_file.read_text())
    completed_layers = int(state.get("completed_layers", 0))
    if completed_layers < 0:
        return 0
    return min(completed_layers * per_stage_updates, total_updates)

seed_durations = []
experiment_started = time.monotonic()
completed_seeds = 0

for i in range(n_indices):
    seed_dir = results_dir / f"seed_{i}"
    metrics_path = seed_dir / "metrics.json"
    checkpoint_dir = seed_dir / "checkpoint"
    log_path = seed_dir / "run.log"

    if metrics_path.exists():
        completed_seeds += 1
        tqdm.write(f"[skip] seed {i:02d} already complete")
        continue

    seed_dir.mkdir(parents=True, exist_ok=True)
    tqdm.write(f"[run ] seed {i:02d}/{n_indices - 1:02d} started")
    command = [
        sys.executable, "-c",
        "import tensorflow as tf; "
        "tf.get_logger().setLevel('ERROR'); "
        "from experiments.run_layerwise import main; main()",
        CONFIG, "--seed-index", str(i),
    ]
    started = time.monotonic()
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            env=run_env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            text=True,
        )
        avg_seed_time = sum(seed_durations) / len(seed_durations) if seed_durations else 0.0
        progress = tqdm(
            total=total_updates,
            initial=checkpoint_step(checkpoint_dir, metrics_path),
            desc=f"seed {i:02d}",
            unit="step",
            dynamic_ncols=True,
            mininterval=0.2,
            maxinterval=1.0,
            smoothing=0.1,
            bar_format=(
                "{desc} |{bar:28}| {percentage:3.0f}% "
                "[{n_fmt}/{total_fmt}] [{elapsed}<{remaining}, {rate_fmt}]"
            ),
        )
        last_step = progress.n

        while process.poll() is None:
            current_step = checkpoint_step(checkpoint_dir, metrics_path)
            if current_step > last_step:
                progress.update(current_step - last_step)
                last_step = current_step
            avg_seed_time = sum(seed_durations) / len(seed_durations) if seed_durations else 0.0
            remaining_seed_time = avg_seed_time * max(n_indices - completed_seeds - 1, 0)
            progress.set_postfix_str(
                f"checkpoint {current_step}/{total_updates} | avg_rem={format_duration(remaining_seed_time)}"
            )
            progress.refresh()
            time.sleep(1)

        return_code = process.wait()
        final_step = checkpoint_step(checkpoint_dir, metrics_path)
        if return_code == 0 and final_step < total_updates:
            progress.update(total_updates - final_step)
        progress.close()

    elapsed = time.monotonic() - started
    if return_code != 0:
        log_tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-12:]
        tqdm.write(f"[fail] seed {i:02d} failed; log: {log_path}")
        print("\n".join(log_tail))
        raise subprocess.CalledProcessError(return_code, command)

    seed_durations.append(elapsed)
    completed_seeds += 1
    average_seed = sum(seed_durations) / len(seed_durations)
    remaining_seeds = n_indices - completed_seeds
    tqdm.write(
        f"[done] seed {i:02d} | time={format_duration(elapsed)} | "
        f"remaining={format_duration(average_seed * remaining_seeds)} | log={log_path}"
    )

print(
    f"Complete: {completed_seeds}/{n_indices} seeds | "
    f"total time={format_duration(time.monotonic() - experiment_started)}",
    flush=True,
)

Layerwise depth 6 | device=cpu | seeds=20 | updates/seed=2500
[skip] seed 00 already complete
[skip] seed 01 already complete
[skip] seed 02 already complete
[skip] seed 03 already complete
[skip] seed 04 already complete
[skip] seed 05 already complete
[skip] seed 06 already complete
[skip] seed 07 already complete
[skip] seed 08 already complete
[skip] seed 09 already complete
[skip] seed 10 already complete
[skip] seed 11 already complete
[skip] seed 12 already complete
[skip] seed 13 already complete
[run ] seed 14/19 started


seed 14 |                            |   0% [0/2500] [00:00<?, ?step/s]

[done] seed 14 | time=41m 35s | remaining=207m 57s | log=results/layerwise/depth_6/seed_14/run.log
[run ] seed 15/19 started


seed 15 |                            |   0% [0/2500] [00:00<?, ?step/s]

[done] seed 15 | time=58m 53s | remaining=200m 58s | log=results/layerwise/depth_6/seed_15/run.log
[run ] seed 16/19 started


...

[done] seed 17 | time=34m 38s | remaining=85m 33s | log=results/layerwise/depth_6/seed_17/run.log
[run ] seed 18/19 started


seed 18 |                            |   0% [0/2500] [00:00<?, ?step/s]

[done] seed 18 | time=32m 47s | remaining=40m 47s | log=results/layerwise/depth_6/seed_18/run.log
[run ] seed 19/19 started


seed 19 |                            |   0% [0/2500] [00:00<?, ?step/s]

[done] seed 19 | time=34m 19s | remaining=0m 00s | log=results/layerwise/depth_6/seed_19/run.log
Complete: 20/20 seeds | total time=238m 15s


## Summary

In [3]:
import json

results_dir = Path("results/layerwise/depth_6")
rows = []
for seed_dir in sorted(results_dir.glob("seed_*")):
    metrics = json.loads((seed_dir / "metrics.json").read_text())
    diag = metrics.get("training_diagnostic", {})
    rows.append((
        metrics.get("seed_index"),
        metrics["test_acc"],
        metrics["test_loss"],
        metrics.get("n_parameters"),
        diag.get("mean_param_grad_variance"),
    ))

rows.sort()
if rows:
    print(f"{len(rows)} completed runs in {results_dir}:")
    print(f"{'seed':>4}  {'test_acc':>8}  {'test_loss':>9}  "
          f"{'n_params':>8}  {'grad_var':>12}")
    for seed, acc, loss, nparams, gv in rows:
        print(f"{seed:>4}  {acc:>8.4f}  {loss:>9.4f}  "
              f"{nparams:>8}  {gv:>12.3e}")
else:
    print(f"No completed runs found in {results_dir}.")


20 completed runs in results/layerwise/depth_6:
seed  test_acc  test_loss  n_params      grad_var
   0    0.8450     0.3904        96     8.698e-03
   1    0.8750     0.3463        96     8.328e-03
   2    0.8750     0.3629        96     5.137e-03
   3    0.8900     0.3004        96     4.648e-03
   4    0.9150     0.2994        96     5.874e-03
   5    0.8600     0.3768        96     7.710e-03
   6    0.8600     0.3626        96     1.061e-02
   7    0.8700     0.3244        96     7.539e-03
   8    0.9000     0.2898        96     5.770e-03
   9    0.8650     0.3440        96     5.569e-03
  10    0.8650     0.3860        96     4.953e-03
  11    0.8350     0.3640        96     6.237e-03
  12    0.8850     0.3252        96     6.818e-03
  13    0.8950     0.3397        96     6.066e-03
  14    0.8350     0.4037        96     5.393e-03
  15    0.8250     0.4114        96     6.436e-03
  16    0.8300     0.3660        96     3.442e-03
  17    0.8600     0.3276        96     5.147e-03
  